<a href="https://colab.research.google.com/github/viswadevadi-spec/fraud-policy-assistant/blob/main/Fraud_policy_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain chromadb sentence-transformers gradio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [2]:
!pip install transformers torch accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
!pip install -q langchain langchain-community chromadb langchain-text-splitters pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
google-adk 1.28.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-c

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/fraud_rag/docs", exist_ok=True)
os.makedirs("/content/drive/MyDrive/fraud_rag/chroma_db", exist_ok=True)

print("Drive mounted and folders ready")

Mounted at /content/drive
Drive mounted and folders ready


In [6]:
import urllib.request
import os

save_dir = "/content/drive/MyDrive/fraud_rag/docs"
os.makedirs(save_dir, exist_ok=True)

# working SEC.gov PDFs (April 2026)
pdfs = {
    "sec_2024_financial_report.pdf":
        "https://www.sec.gov/files/sec-2024-agency-financial-report.pdf",

    "sec_2026_exam_priorities.pdf":
        "https://www.sec.gov/files/2026-exam-priorities.pdf",

    "sec_whistleblower_2025.pdf":
        "https://www.sec.gov/files/fy25-annual-whistleblower-report.pdf",

    "sec_regulation_sp_compliance.pdf":
        "https://www.sec.gov/files/rules/final/2024/regulation-s-p-small-entity-compliance-guide.pdf",
}

headers = {'User-Agent': 'Mozilla/5.0 fraud-rag-project research@example.com'}

for filename, url in pdfs.items():
    path = f"{save_dir}/{filename}"
    if os.path.exists(path) and os.path.getsize(path) > 1000:
        print(f"  Already exists: {filename}")
        continue
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=30) as response:
            with open(path, 'wb') as f:
                f.write(response.read())
        size_kb = os.path.getsize(path) / 1024
        print(f" Downloaded: {filename} ({size_kb:.0f} KB)")
    except Exception as e:
        print(f" Failed: {filename} → {e}")

print("\n Files in docs folder:")
for f in os.listdir(save_dir):
    size = os.path.getsize(f"{save_dir}/{f}") / 1024
    print(f"   {f} ({size:.0f} KB)")

  Already exists: sec_2024_financial_report.pdf
  Already exists: sec_2026_exam_priorities.pdf
  Already exists: sec_whistleblower_2025.pdf
  Already exists: sec_regulation_sp_compliance.pdf

 Files in docs folder:
   sec_2024_financial_report.pdf (6654 KB)
   sec_2026_exam_priorities.pdf (438 KB)
   sec_whistleblower_2025.pdf (222 KB)
   sec_regulation_sp_compliance.pdf (159 KB)


In [7]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the documents from the folder
loader = PyPDFDirectoryLoader("/content/drive/MyDrive/fraud_rag/docs/")
docs = loader.load()
print(f" Loaded {len(docs)} pages")

# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(docs)
print(f" Created {len(chunks)} chunks")

 Loaded 221 pages
 Created 1221 chunks


In [8]:
import chromadb
from sentence_transformers import SentenceTransformer

# Embed
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [c.page_content for c in chunks]
embeddings = embed_model.encode(texts, batch_size=32, show_progress_bar=True)

# Store — persisted to Drive, survives session restarts
client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/fraud_rag/chroma_db"
)
collection = client.get_or_create_collection(
    name="fraud_policy",
    metadata={"hnsw:space": "cosine"}
)

# Only add if collection is empty
if collection.count() == 0:
    collection.add(
        documents=texts,
        embeddings=embeddings.tolist(),
        ids=[f"doc_{i}" for i in range(len(texts))]
    )
    print(f" Stored {len(texts)} chunks in ChromaDB")
else:
    print(f" ChromaDB already has {collection.count()} chunks — skipping")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

 ChromaDB already has 1221 chunks — skipping


In [9]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import bitsandbytes as bnb




In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Using Qwen2.5-3B which is open and fits well in Colab T4
model_id = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f" Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# We name this 'model' to match your query_fraud_assistant function
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="cuda",
    low_cpu_mem_usage=True
)
model.eval()

print(" Model loaded as 'model' — no login needed")

 Loading Qwen/Qwen2.5-3B-Instruct...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

 Model loaded as 'model' — no login needed


In [11]:
def query_fraud_assistant(question: str) -> str:
    # Step 1: Embed the query
    q_embedding = embed_model.encode([question])[0].tolist()

    # Step 2: Retrieve top-5 chunks
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=5
    )
    context = "\n\n---\n\n".join(results['documents'][0])

    # Step 3: Build prompt
    prompt = f"""You are a fraud policy expert. Answer using ONLY the context below.
If the answer isn't in the context, say "I don't know." Don't hallucinate.

Context:
{context}

Question: {question}
Answer (with source references):"""

    # Step 4: Tokenize + Generate (Using the 'model' variable)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens
    answer = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return answer.strip()

# Quick test
if 'model' in globals():
    print(query_fraud_assistant("What is a Ponzi scheme?"))
else:
    print("Load the model in the cell above first!")

A Ponzi scheme is a fraudulent scheme in which returns are paid to investors with funds from new investors rather than from profits. This type of scheme is characterized by the promise of high returns in a short period of time, often through the recruitment of new investors. The hallmark of Ponzi schemes is their reliance on ongoing fundraising to sustain payouts, making them unsustainable in the long term. This behavior is in direct opposition to legitimate financial practices, where returns are generated from actual investment profits. I don't know. 

The context does not provide a definition of a Ponzi scheme. However, the question has been answered based on general knowledge and the provided context, which mentions that Ponzi schemes involve paying returns to investors with funds from new investors rather than from profits. The context also notes that Ponzi schemes rely on ongoing fundraising to sustain payouts, making them unsustainable in the long term. This description aligns wi

In [12]:
import gradio as gr

def respond(question):
    if not question.strip():
        return "Please ask a question."
    return query_fraud_assistant(question)

demo = gr.Interface(
    fn=respond,
    inputs=gr.Textbox(
        label="Ask about fraud policy...",
        placeholder="e.g. What are red flags for securities fraud?",
        lines=3
    ),
    outputs=gr.Textbox(label="Answer", lines=10),
    title="🔍 Fraud Policy Assistant",
    description="RAG-powered compliance Q&A | Llama-3.2-3B + ChromaDB + SEC Docs",
    examples=[
        ["What is a Ponzi scheme?"],
        ["What are common signs of securities fraud?"],
        ["How does the SEC detect fraud?"]
    ]
)

demo.launch(share=True)  # share=True gives public URL for portfolio

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1bf05aaf5272fd977f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
